# 01 · Exploratory data analysis: NIH ChestX-ray14

Goals:
1. Show that the task is **multi-label**: many images carry more than one condition.
2. Quantify the **class imbalance** that the loss functions and thresholds must handle.
3. Check that the train/val/test split is **patient-level**, with no leakage.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# Find the project code (folder that contains src/): locally, or on Kaggle as an uploaded dataset.
candidates = [Path.cwd().parent, Path.cwd(), Path('/kaggle/working/project'), Path('/kaggle/input/cxr-capstone-code')]
PROJECT_DIR = next(p for p in candidates if (p / 'src' / 'labels.py').exists())
sys.path.insert(0, str(PROJECT_DIR)); print('code from', PROJECT_DIR)

# Kaggle: /kaggle/input/data   ·   local: wherever you extracted the dataset
DATA_DIR = os.environ.get('NIH_DATA_DIR', '/kaggle/input/data')

from src.labels import CLASSES
from src.dataset import load_metadata, make_splits, load_image
df = load_metadata(DATA_DIR)
print(len(df), 'images ·', df.patient_id.nunique(), 'patients')
df.head()

## Class imbalance

In [ ]:
counts = df[CLASSES].sum().sort_values()
no_finding = (df[CLASSES].sum(1) == 0).mean()
ax = (counts / len(df) * 100).plot.barh(figsize=(8, 6))
ax.set_xlabel('% of images'); ax.set_title(f'Prevalence per condition  (No Finding = {no_finding:.1%})')
for i, v in enumerate(counts): ax.text(v / len(df) * 100 + 0.2, i, f'{int(v)}', va='center', fontsize=8)
plt.tight_layout(); plt.show()
print('Imbalance ratio (most / least common):', round(counts.max() / counts.min(), 1))

## Multi-label: number of conditions per image

In [ ]:
n_labels = df[CLASSES].sum(1).astype(int)
print(n_labels.value_counts().sort_index())
print(f'{(n_labels >= 2).mean():.1%} of all images, and {(n_labels[n_labels > 0] >= 2).mean():.1%} of abnormal images, have 2+ conditions')
n_labels.value_counts().sort_index().plot.bar(title='Conditions per image'); plt.show()

## Co-occurrence: P(column | row)

In [ ]:
Y = df[CLASSES].values
co = Y.T @ Y
cond = co / np.clip(np.diag(co)[:, None], 1, None)  # row-normalised: P(col | row)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cond, cmap='viridis', vmax=0.5)
ax.set_xticks(range(14), CLASSES, rotation=90); ax.set_yticks(range(14), CLASSES)
for i in range(14):
    for j in range(14):
        if i != j and cond[i, j] > 0.15: ax.text(j, i, f'{cond[i, j]:.2f}', ha='center', va='center', fontsize=7, color='w')
plt.colorbar(im); ax.set_title('P(column condition | row condition)'); plt.tight_layout(); plt.show()

## Patient-level split

In [ ]:
train_df, val_df, test_df = make_splits(df, DATA_DIR)   # asserts no patient overlap
split = pd.DataFrame({s: d[CLASSES].mean() for s, d in [('train', train_df), ('val', val_df), ('test', test_df)]})
print({k: len(v) for k, v in [('train', train_df), ('val', val_df), ('test', test_df)]})
split.style.format('{:.3f}')

## Sample images

In [ ]:
sample = df[df[CLASSES].sum(1) >= 2].sample(8, random_state=0)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(load_image(r.path).convert('L'), cmap='gray'); ax.axis('off')
    ax.set_title(r.findings.replace('|', '\n'), fontsize=9)
plt.tight_layout(); plt.show()

## Radiologist bounding boxes (used later to evaluate Grad-CAM)

In [ ]:
from src.localization import load_bboxes
from PIL import ImageDraw
boxes = load_bboxes(DATA_DIR)
print(boxes.label.value_counts())
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (_, b) in zip(axes, boxes.groupby('label').head(1).head(4).iterrows()):
    img = load_image(b.path); ImageDraw.Draw(img).rectangle([b.x, b.y, b.x + b.w, b.y + b.h], outline=(255, 0, 0), width=6)
    ax.imshow(img); ax.set_title(b.label); ax.axis('off')
plt.show()